In [68]:
import depthai as dai
import numpy as np
import json
import ast



In [3]:
def build_homogeneous(rotation_matrix, translation_vector):
    T_camera_to_base_effector = np.eye(4)
    T_camera_to_base_effector[:3, :3] = rotation_matrix
    T_camera_to_base_effector[:3, 3] = translation_vector.reshape(3)
    return T_camera_to_base_effector

def convert_coordinates(x ,y ,z, homogeneous_matrix): # X Y Z coordinates that should be translated into robot frame coordinates
    obj_camera_coordinates = np.array([x, y, z])
    obj_camera_coordinates_homo = np.append(obj_camera_coordinates, [1])  # Convert object coordinates to homogeneous coordinates
    obj_base_effector_coordinates_homo = homogeneous_matrix.dot(obj_camera_coordinates_homo)
    obj_base_coordinates = obj_base_effector_coordinates_homo[:3]  
    
    #return list(map(int, obj_base_coordinates)) # Uncomment this line if you want to send integers instead of floats
    return np.around(obj_base_coordinates,2).tolist() # Use this to get a list of new coordinates, chane the number to get the number of decimal numbers

def estimate_rigid_transform(camera_points, robot_points):
    cam = np.asarray(camera_points, dtype=np.float64)
    rob = np.asarray(robot_points,  dtype=np.float64)
    assert cam.shape == rob.shape and cam.shape[1] == 3 and cam.shape[0] >= 3, "Value error, differing amount of coordinates, Camera:" + str(len(cam)) + " Robot:"+ str(len(rob))
    

    camera_centroid = cam.mean(axis=0)
    robot_centroid  = rob.mean(axis=0)
    camera_centered = cam - camera_centroid
    robot_centered  = rob - robot_centroid

    cross_covariance = camera_centered.T @ robot_centered
    U, s, Vt = np.linalg.svd(cross_covariance)
    V = Vt.T

    # Ensure of proper rotation (det=+1)
    det_correction = np.sign(np.linalg.det(V @ U.T))
    rotation_matrix = V @ np.diag([1.0, 1.0, det_correction]) @ U.T

    translation_vector = robot_centroid - rotation_matrix @ camera_centroid
    return rotation_matrix, translation_vector

def extract_data(file):
    
    float_list=[]
    with open(file, "r") as f:
        lines = f.readlines()
        for i in lines:
            x = json.loads(i)
            float_list.append(x)
    return list(float_list)

In [4]:
cam_coords = extract_data("saved_coordinates.txt")
robot_coords = extract_data("robo_coords.txt")

print(cam_coords)
print(robot_coords)



[[18.804306030273438, -8.065057754516602, 775.48779296875], [-253.22769165039062, 37.41748046875, 981.2294921875], [-227.116943359375, 149.2969970703125, 1285.5679931640625], [24.686628341674805, 39.52834701538086, 950.2024536132812], [-72.41410064697266, 160.78977966308594, 1306.5283203125], [-209.0472869873047, 39.44023513793945, 989.305419921875], [65.744384765625, 32.89694595336914, 903.7639770507812], [-150.45704650878906, 61.27862548828125, 1010.0891723632812], [-35.62618637084961, 162.66688537597656, 1285.5679931640625], [-23.708532333374023, 7.41449499130249, 855.5203247070312]]
[[394.163, 360.014, 45.2468], [598.309, 95.9641, 44.5703], [447.773, -180.739, 46.378], [318.487, 198.147, 37.1674], [280.026, -154.354, 38.4586], [548.93, 95.9352, 39.5118], [301.78, 265.922, 42.8517], [479.935, 73.9366, 44.6034], [252.326, -131.682, 42.2974], [411, 267.247, 37.757]]


In [5]:
R, t = estimate_rigid_transform(cam_coords, robot_coords)
print("R =\n", R)
print("t =", t)
homogeneous = build_homogeneous(R,t)
print(homogeneous)
print(convert_coordinates(100,100,100, homogeneous))

R =
 [[-0.93480345 -0.16182031 -0.31615929]
 [ 0.3513136  -0.29054895 -0.89003374]
 [ 0.05216579 -0.94307767  0.32845583]]
t = [ 660.7177316  1059.75511264 -228.96709574]
[[-9.34803448e-01 -1.61820315e-01 -3.16159293e-01  6.60717732e+02]
 [ 3.51313597e-01 -2.90548946e-01 -8.90033745e-01  1.05975511e+03]
 [ 5.21657914e-02 -9.43077672e-01  3.28455833e-01 -2.28967096e+02]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]
[519.44, 976.83, -285.21]


In [ ]:

with open('Rotation.txt', 'a') as f:
    for i in R:
        f.write("%s\n" % i) #saves the camera coordinates to a .txt file
    f.write("\n")


with open('Translation.txt', 'a') as f2:
    f2.write("%s\n" % t) #saves the camera coordinates to a .txt file
    f2.write("\n")
    

In [ ]:
with open('RT.txt', 'r') as skib:
    for i in skib:
        print(i.strip("[]\n "))
        



-0.92623124 -0.13817656 -0.35071774
0.37559562 -0.25932592 -0.88976289
0.03199418 -0.95585423  0.29209428
702.4814346  1077.44936097 -191.34991418

-0.91848099 -0.10433763 -0.38145293
0.3948439  -0.29599143 -0.86976282
-0.02215781 -0.94947498  0.31305958
734.1088165  1058.39903609 -228.29970396

-0.93310337 -0.17254745 -0.3155083
0.35288579 -0.27050023 -0.8957127
0.06920788 -0.94713093  0.31329422
662.36055108 1076.00707192 -208.33337613

-0.91716034 -0.28661844 -0.27688769
0.35331986 -0.26342868 -0.89764715
0.18434206 -0.92111628  0.34287432
629.14879567 1070.31684155 -226.04853855

-0.93480345 -0.16182031 -0.31615929
0.3513136  -0.29054895 -0.89003374
0.05216579 -0.94307767  0.32845583
660.7177316  1059.75511264 -228.96709574

-0.93480345 -0.16182031 -0.31615929
0.3513136  -0.29054895 -0.89003374
0.05216579 -0.94307767  0.32845583
660.7177316  1059.75511264 -228.96709574

-0.93480345 -0.16182031 -0.31615929
0.3513136  -0.29054895 -0.89003374
0.05216579 -0.94307767  0.32845583
660.717

In [78]:
with open('Rotation.txt', 'r') as f3: 
    for i in f3:
        np.fromstring(i, sep=' ')

C:\Users\aar20007\AppData\Local\Temp\ipykernel_5904\2781745835.py:3: DeprecationWarning: string or file could not be read to its end due to unmatched data; this will raise a ValueError in the future.
  np.fromstring(i, sep=' ')


In [84]:
test = np.fromfile("Rotation.txt")
print(test)

[5.93627329e-038 3.25266779e-086 1.08601738e-071 1.25603328e-071
 3.61170779e+130 1.08624104e-071 5.59871073e-067 1.58292198e-153
 3.93517551e-062 8.06672248e-153 2.31697090e-052 4.01147276e-057
 6.04709998e-154 2.21379907e-052]


In [ ]:
def parse_file(path):
    R_list = []
    t_list = []

    with open(path, "r") as f:
        block = []
        for line in f:
            line = line.strip()
            if not line:
                continue
            block.append(line)

            if len(block) == 4:
                R = []
                for i in range(3):
                    R.append([float(x) for x in block[i].replace("[","").replace("]","").split()])
                R = np.array(R)
                t = np.array([float(x) for x in block[3].replace("[","").replace("]","").split()])
                R_list.append(R)
                t_list.append(t)
                block = []

    return np.array(R_list), np.array(t_list)


def average_rotations(Rs):
    M = np.zeros((3, 3))
    for R in Rs:
        M += R
    M /= len(Rs)
    return R_avg


def average_translations(ts):
    return np.mean(ts, axis=0)


def save_result(path, R_avg, t_avg):
    with open(path, "w") as f:
        f.write("Average Rotation Matrix:\n")
        f.write(str(R_avg) + "\n\n")
        f.write("Average Translation Vector:\n")
        f.write(str(t_avg) + "\n")



input_file = "RT.txt"
output_file = "AveragedOutput.txt"

R_list, t_list = parse_file(input_file)

R_avg = average_rotations(R_list)
t_avg = average_translations(t_list)

save_result(output_file, R_avg, t_avg)

print("Average Rotation:\n", R_avg)
print("Average Translation:\n", t_avg)


Average Rotation:
 [[-0.93030928 -0.16980799 -0.32509982]
 [ 0.36183316 -0.27987743 -0.88923866]
 [ 0.06001172 -0.94489887  0.32181473]]
Average Translation:
 [ 672.89325609 1065.91966406 -220.13326001]
